# Markov Chain Baseline for Music Generation

This notebook implements a simple Markov chain baseline for comparison with deep learning models.

In [ ]:
import os
import sys
import numpy as np
import pretty_midi
from collections import defaultdict, Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import random

# Add src to path
sys.path.append('../src')

from config import *

## 1. Markov Chain Model

A simple first-order Markov chain that models transitions between pitch classes.

In [ ]:
class MarkovChainGenerator:
    """
    First-order Markov chain for music generation.
    Models transitions between pitch classes (0-11).
    """
    
    def __init__(self, order=1):
        """
        Initialize Markov chain.
        
        Args:
            order: Order of Markov chain (1 = first-order)
        """
        self.order = order
        self.transitions = defaultdict(Counter)
        self.start_states = Counter()
    
    def train(self, midi_paths):
        """
        Train Markov chain on MIDI files.
        
        Args:
            midi_paths: List of paths to MIDI files
        """
        print(f"Training Markov chain (order={self.order})...")
        
        for path in tqdm(midi_paths):
            try:
                # Load MIDI
                midi = pretty_midi.PrettyMIDI(path)
                
                # Extract pitch classes from all instruments
                notes = []
                for instrument in midi.instruments:
                    if not instrument.is_drum:
                        notes.extend([(n.start, n.pitch % 12) for n in instrument.notes])
                
                # Sort by time
                notes.sort(key=lambda x: x[0])
                pitch_classes = [pc for _, pc in notes]
                
                if len(pitch_classes) < 2:
                    continue
                
                # Record start state
                self.start_states[pitch_classes[0]] += 1
                
                # Record transitions
                for i in range(len(pitch_classes) - 1):
                    current = pitch_classes[i]
                    next_pc = pitch_classes[i + 1]
                    self.transitions[current][next_pc] += 1
            
            except Exception as e:
                continue
        
        print(f"Trained on {len(midi_paths)} files")
        print(f"Unique states: {len(self.transitions)}")
    
    def generate(self, length=100, start_state=None):
        """
        Generate sequence using Markov chain.
        
        Args:
            length: Number of notes to generate
            start_state: Starting pitch class (None = random)
            
        Returns:
            List of pitch classes
        """
        if not self.transitions:
            raise ValueError("Model not trained")
        
        # Choose start state
        if start_state is None:
            if self.start_states:
                start_state = random.choices(
                    list(self.start_states.keys()),
                    weights=list(self.start_states.values())
                )[0]
            else:
                start_state = random.choice(list(self.transitions.keys()))
        
        sequence = [start_state]
        current = start_state
        
        # Generate sequence
        for _ in range(length - 1):
            if current not in self.transitions or not self.transitions[current]:
                # Dead end - choose random state
                current = random.choice(list(self.transitions.keys()))
            else:
                # Sample next state
                next_states = list(self.transitions[current].keys())
                weights = list(self.transitions[current].values())
                current = random.choices(next_states, weights=weights)[0]
            
            sequence.append(current)
        
        return sequence
    
    def to_midi(self, pitch_classes, output_path, tempo=120, duration=0.5):
        """
        Convert pitch class sequence to MIDI file.
        
        Args:
            pitch_classes: List of pitch classes (0-11)
            output_path: Path to save MIDI file
            tempo: Tempo in BPM
            duration: Note duration in seconds
        """
        midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
        piano = pretty_midi.Instrument(program=0)
        
        # Convert pitch classes to actual pitches (middle octave)
        current_time = 0.0
        for pc in pitch_classes:
            pitch = 60 + pc  # Middle C + pitch class
            note = pretty_midi.Note(
                velocity=80,
                pitch=pitch,
                start=current_time,
                end=current_time + duration
            )
            piano.notes.append(note)
            current_time += duration
        
        midi.instruments.append(piano)
        midi.write(output_path)
        print(f"Saved MIDI to: {output_path}")

## 2. Load Training Data

In [ ]:
# Get list of training MIDI files
import pandas as pd

csv_path = os.path.join(RAW_MIDI_DIR, 'maestro-v3.0.0.csv')
df = pd.read_csv(csv_path)
df['basename'] = df['midi_filename'].apply(os.path.basename)
actual_files = [f for f in os.listdir(RAW_MIDI_DIR) if f.endswith('.midi')]
df = df[df['basename'].isin(actual_files)].copy()
df['midi_path'] = df['basename'].apply(lambda x: os.path.join(RAW_MIDI_DIR, x))

train_df = df[df['split'] == 'train'].copy()

# Use subset for faster training (optional)
train_paths = train_df['midi_path'].tolist()[:100]  # Use first 100 files

print(f"Training on {len(train_paths)} MIDI files")

## 3. Train Markov Chain

In [ ]:
# Initialize and train model
markov = MarkovChainGenerator(order=1)
markov.train(train_paths)

## 4. Visualize Transition Matrix

In [ ]:
# Create transition matrix
transition_matrix = np.zeros((12, 12))

for state, transitions in markov.transitions.items():
    total = sum(transitions.values())
    for next_state, count in transitions.items():
        transition_matrix[state, next_state] = count / total

# Plot
plt.figure(figsize=(10, 8))
plt.imshow(transition_matrix, cmap='Blues', aspect='auto')
plt.colorbar(label='Transition Probability')
plt.xlabel('Next Pitch Class')
plt.ylabel('Current Pitch Class')
plt.title('Markov Chain Transition Matrix')

# Add pitch class labels
pitch_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
plt.xticks(range(12), pitch_names)
plt.yticks(range(12), pitch_names)

plt.tight_layout()
plt.show()

## 5. Generate Music

In [ ]:
# Generate sequences
num_samples = 5
sequence_length = 200

os.makedirs(GENERATED_MIDIS_DIR, exist_ok=True)

for i in range(num_samples):
    # Generate sequence
    sequence = markov.generate(length=sequence_length)
    
    # Save as MIDI
    output_path = os.path.join(GENERATED_MIDIS_DIR, f'markov_baseline_{i+1}.midi')
    markov.to_midi(sequence, output_path)
    
    print(f"Generated sample {i+1}: {len(sequence)} notes")

print(f"\nGenerated {num_samples} samples")

## 6. Analyze Generated Sequences

In [ ]:
# Generate multiple sequences for analysis
sequences = [markov.generate(length=200) for _ in range(100)]

# Compute pitch class distribution
all_pitches = [pc for seq in sequences for pc in seq]
pitch_counts = Counter(all_pitches)

# Plot distribution
plt.figure(figsize=(10, 5))
pitch_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
counts = [pitch_counts[i] for i in range(12)]

plt.bar(range(12), counts, color='#4C72B0', alpha=0.7)
plt.xlabel('Pitch Class')
plt.ylabel('Frequency')
plt.title('Pitch Class Distribution (Markov Baseline)')
plt.xticks(range(12), pitch_names)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nPitch class distribution:")
for i, name in enumerate(pitch_names):
    print(f"  {name}: {counts[i]} ({counts[i]/sum(counts)*100:.1f}%)")

## 7. Compute Baseline Metrics

In [ ]:
# Compute entropy of pitch class distribution
probs = np.array(counts) / sum(counts)
entropy = -np.sum(probs * np.log2(probs + 1e-10))

print(f"Pitch class entropy: {entropy:.3f} bits")
print(f"Max entropy (uniform): {np.log2(12):.3f} bits")
print(f"Normalized entropy: {entropy / np.log2(12):.3f}")

# Compute transition entropy
transition_entropies = []
for state in range(12):
    if state in markov.transitions:
        trans = markov.transitions[state]
        total = sum(trans.values())
        probs = np.array([trans[i] / total for i in range(12)])
        ent = -np.sum(probs * np.log2(probs + 1e-10))
        transition_entropies.append(ent)

avg_transition_entropy = np.mean(transition_entropies)
print(f"\nAverage transition entropy: {avg_transition_entropy:.3f} bits")

## Summary

The Markov chain baseline provides a simple probabilistic model for music generation:
- Models transitions between pitch classes
- Fast training and generation
- Limited expressiveness (no rhythm, dynamics, or long-term structure)
- Useful as a baseline for comparison with deep learning models

**Limitations:**
- Only models pitch, not rhythm or timing
- No long-term dependencies (first-order)
- No polyphony (single note at a time)
- No dynamics or articulation